# Time Series Analysis Practical Session

## Goal
In this session, we will apply the concepts of Time Series Analysis to a real-world dataset. We will cover:
1.  **Decomposition**: Breaking down a series into Trend, Seasonality, and Noise.
2.  **Stationarity**: Testing if statistical properties are constant over time (ADF, KPSS).
3.  **Splitting Strategies**: Understanding why random splitting fails and how to do it right.
4.  **Diagnostics**: Using ACF and PACF plots to identify potential model parameters.
5.  **Forecasting with Multiple Models**:
    - **SARIMA**: Statistical approach.
    - **Random Forest**: Machine Learning approach (using lag features).
    - **Prophet**: Facebook's additive regression model.
6.  **Error Analysis**: Comparing models using MAE, RMSE, and MAPE.

## Dataset
We will use the classic **Airline Passengers** dataset, which shows the monthly number of US airline passengers from 1949 to 1960.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, train_test_split

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)

## Task 1: Load and Inspect Data

1. Load the dataset from the provided URL.
2. Ensure the 'Month' column is parsed as dates and set as the index.
3. Plot the 'Passengers' series to visualize the data.

**URL**: `https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv`

In [ ]:
# TODO: Load the dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
# data = ...

# TODO: Plot the time series


---

## Task 2: Decomposition

We will decompose the time series to verify your observations.

### 2.1 STL Decomposition
Use `STL` from `statsmodels` to automatically decompose the series into Trend, Seasonal, and Residual components.

In [ ]:
# TODO: Perform STL Decomposition
# stl = STL(..., seasonal=13)
# result = stl.fit()

# TODO: Plot the components (result.plot())


---

## Task 3: Splitting Strategies - Good vs Bad

In standard Machine Learning (e.g., classifying images), we often split data randomly. **In Time Series, this is a disaster!**

### 3.1 The Wrong Way: Random Split
If we split randomly, we might train on data from 1955 and 1957 to predict 1956. This is **Data Leakage** (using the future to predict the past).

In [ ]:
# Visualizing Random Split (BAD)
train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=42)

plt.figure(figsize=(10, 4))
plt.scatter(train_idx, data.loc[train_idx, 'Passengers'], color='blue', label='Train (Random)', s=10)
plt.scatter(test_idx, data.loc[test_idx, 'Passengers'], color='red', label='Test (Random)', s=10)
plt.title('BAD: Random Split (Data Leakage!)')
plt.legend()
plt.show()

### 3.2 The Right Way: Time-Based Split
We must always split dependent on time: Train on the past, Test on the future.

In [ ]:
# Visualizing Time-Based Split (GOOD)
split_date = data.index[-24]
train_data = data[data.index < split_date]
test_data = data[data.index >= split_date]

plt.figure(figsize=(10, 4))
plt.plot(train_data.index, train_data['Passengers'], color='blue', label='Train (Past)')
plt.plot(test_data.index, test_data['Passengers'], color='red', label='Test (Future)')
plt.axvline(split_date, color='black', linestyle='--')
plt.title('GOOD: Time-Based Split')
plt.legend()
plt.show()

### 3.3 Cross-Validation with TimeSeriesSplit
Use `TimeSeriesSplit` from `sklearn` to create multiple train/test folds that respect time order.

In [ ]:
def plot_cv_indices(cv, X, n_splits, lw=10):
    """Create a sample plot for indices of a cross-validation object."""
    fig, ax = plt.subplots()
    
    for ii, (train, test) in enumerate(cv.split(X)):
        indices = np.array([np.nan] * len(X))
        indices[train] = 1
        indices[test] = 0
        ax.scatter(range(len(indices)), [ii + .5] * len(indices),
                   c=indices, marker='_', lw=lw, cmap=plt.cm.coolwarm,
                   vmin=-.2, vmax=1.2)

    ax.set_yticklabels(list(range(n_splits)))
    ax.set_title('TimeSeriesSplit Behavior')
    ax.set_ylabel('CV Iteration')
    ax.set_xlabel('Time Index')
    ax.legend(["Testing", "Training"], loc=(1.02, .8))
    return ax

# TODO: Initialize TimeSeriesSplit(n_splits=5)
# tscv = TimeSeriesSplit(n_splits=5)

# TODO: Visualize the splits using the helper function above
# plot_cv_indices(tscv, data['Passengers'], 5)
# plt.show()

---

## Task 4: Stationarity & Diagnostics

1. Check stationarity (ADF/KPSS).
2. Apply differencing to make it stationary.
3. Plot ACF/PACF.

In [ ]:
# TODO: Create 'Diff' column (difference of Passengers)

# TODO: Plot ACF and PACF of the differenced data


---

## Task 5: Forecasting Setup

We will split the data into **Train** and **Test** sets to evaluate our models.
- **Test Set**: Last 24 months (2 years).
- **Train Set**: Everything before that.

We will also define an error metric function.

In [ ]:
# TODO: Split data
test_months = 24
# train = ...
# test = ...

print(f"Train size: {len(train)}")
print(f"Test size: {len(test)}")

def evaluate_forecast(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"--- {model_name} ---")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAPE: {mape:.2f}%")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

---

## Task 6: Model 1 - SARIMA

Fit a SARIMA model on the **Train** set and forecast the **Test** set.
Suggested Order: `(0, 1, 1) x (0, 1, 1, 12)`

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# TODO: Fit SARIMA on Train
# model_sarima = SARIMAX(..., order=(0,1,1), seasonal_order=(0,1,1,12))
# result_sarima = model_sarima.fit()

# TODO: Forecast (steps=len(test))
# preds_sarima = ...

# Evaluate
# evaluate_forecast(test['Passengers'], preds_sarima, "SARIMA")

---

## Task 7: Model 2 - Random Forest (Machine Learning)

To use ML models for time series, we need to convert the data into a supervised learning format (X, y).
We create **Lag Features** (e.g., value at t-1, t-2, t-12) to predict value at t.

In [ ]:
def create_lag_features(df, lags=[1, 2, 3, 11, 12, 13]):
    df_ml = df.copy()
    for l in lags:
        df_ml[f'lag_{l}'] = df_ml['Passengers'].shift(l)
    return df_ml.dropna()

# TODO: Create features
# df_ml = create_lag_features(data)

# TODO: Split into Train/Test (respecting time order!)
# X = df_ml.drop('Passengers', axis=1)
# y = df_ml['Passengers']
# X_train, X_test = ...
# y_train, y_test = ...


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: Train Random Forest
# model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
# model_rf.fit(X_train, y_train)

# TODO: Predict
# preds_rf = model_rf.predict(X_test)

# Evaluate
# evaluate_forecast(y_test, preds_rf, "Random Forest")

---

## Task 8: Model 3 - Prophet

Prophet requires a specific dataframe format:
- `ds`: Datetime column
- `y`: Target value column

In [ ]:
# !pip install prophet
from prophet import Prophet

# TODO: Prepare data for Prophet (reset index, rename columns)
# prop_data = ...
# prop_train = ... (split based on date)
# prop_test = ...

# TODO: Fit Prophet
# model_prophet = Prophet()
# model_prophet.fit(prop_train)

# TODO: Predict
# future = model_prophet.make_future_dataframe(periods=len(prop_test), freq='MS')
# forecast_prophet = model_prophet.predict(future)
# preds_prophet = forecast_prophet['yhat'].tail(len(prop_test))

# Evaluate
# evaluate_forecast(prop_test['y'], preds_prophet, "Prophet")

### Prophet Diagnostics
Prophet has built-in plotting for components (trend, yearly seasonality, weekly seasonality).
Also visualize the forecast with confidence intervals.

In [ ]:
# TODO: Plot Prophet Components
# model_prophet.plot_components(forecast_prophet)

# TODO: Plot Forecast with confidence intervals
# model_prophet.plot(forecast_prophet)


---

## Task 9: Model Comparison

Visualize the **Full History** (Train), **Actual Future** (Test), and **Predicted** values for all models.

In [ ]:
# TODO: Plot Train + Test + Predictions

# plt.plot(train.index, train['Passengers'], label='Train (History)', color='gray')
# plt.plot(test.index, test['Passengers'], label='Test (Actual)', color='black', linewidth=2)

# plt.plot(test.index, preds_sarima, label='SARIMA', linestyle='--')
# plt.plot(test.index, preds_rf, label='Random Forest', linestyle='--')
# plt.plot(test.index, preds_prophet, label='Prophet', linestyle='--')

# plt.legend()
# plt.show()